# wandb-init-run — worked example 3: Call wandb.init inside a sweep agent function

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-init-run`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When running a wandb sweep, the sweep agent calls your training function once per trial. Inside that function, you call `wandb.init()` WITHOUT passing `config=` — instead, you let wandb populate `wandb.config` automatically from the sweep's sampled values. After `init`, you read back the sampled hyperparameters from `wandb.config`.

## Worked solution

**Step 1 — init without explicit config.**
Inside the sweep agent's callback, we call `wandb.init(project=..., name=...)` with only `project` and `name`. We deliberately omit `config=` because the sweep controller has already placed the sampled hparams into wandb's run config.

**Step 2 — read from wandb.config.**
After `init`, `wandb.config` acts like a dict of the sampled hyperparameters. We read them back with `dict(wandb.config)` and apply them to our args dataclass.

**Step 3 — proceed to training.**
With args now reflecting the sweep's chosen values, we run the actual training. This is the entry point that the sweep controller invokes in parallel across multiple agents.

In [ ]:
import sys
from unittest.mock import MagicMock
from dataclasses import dataclass
sys.modules.setdefault('wandb', MagicMock())
import wandb

@dataclass
class Args:
    lr: float = 1e-3
    batch_size: int = 64
    wandb_project: str = 'sweep-run'
    wandb_name: str = 'agent-trial'

def sweep_agent_fn(args):
    """
    Function the sweep agent calls. Does NOT pass config= to init —
    instead reads back sampled hparams from wandb.config.
    """
    # Open run — NO config= kwarg here
    wandb.init(project=args.wandb_project, name=args.wandb_name)
    
    # Read sweep-sampled hparams back from wandb.config
    sampled = dict(wandb.config)
    for k, v in sampled.items():
        if hasattr(args, k):
            setattr(args, k, v)
    
    # Proceed to training with updated args
    fake_loss = args.lr * 1000  # stand-in for real training
    wandb.log({'loss': fake_loss})
    wandb.finish()
    return args

# Exercise it — simulate what sweep controller does
wandb.init.reset_mock()
wandb.config = {'lr': 2e-4, 'batch_size': 128}  # sweep sampled these
args = Args()
result = sweep_agent_fn(args)
print('Init called without config=:', 'config' not in (wandb.init.call_args.kwargs or {}))
print('Args after sweep update:', result)